## Step 1: Import Required Libraries

In [1]:
import os
import ollama
from pathlib import Path
from typing import List, Dict
import chromadb
from chromadb.config import Settings
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 2: Setup Paths and Configuration

In [2]:
# Configuration
DATA_FOLDER = Path("data")
DB_FOLDER = Path("chroma_db")
CHUNK_SIZE = 500  # words per chunk
CHUNK_OVERLAP = 100  # overlapping words
MODEL_NAME = "mistral"  # Your Ollama model

# Create data folder if it doesn't exist
DATA_FOLDER.mkdir(exist_ok=True)

print(f"📁 Data folder: {DATA_FOLDER.absolute()}")
print(f"💾 Database folder: {DB_FOLDER.absolute()}")
print(f"🤖 LLM Model: {MODEL_NAME}")

📁 Data folder: /Users/sarimkhan/Downloads/AAA/SAAS/rag/data
💾 Database folder: /Users/sarimkhan/Downloads/AAA/SAAS/rag/chroma_db
🤖 LLM Model: mistral


## Step 3: Load Embedding Model

In [3]:
print("Loading embedding model... (this may take a minute first time)")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded!")
print(f"   Model dimension: {embedding_model.get_sentence_embedding_dimension()}")

Loading embedding model... (this may take a minute first time)
✅ Embedding model loaded!
   Model dimension: 384


## Step 4: Initialize ChromaDB


In [4]:
print("Initializing ChromaDB...")
# Disable telemetry to avoid errors
chroma_client = chromadb.PersistentClient(
    path=str(DB_FOLDER),
    settings=Settings(anonymized_telemetry=False)
)

# Get or create collection
collection = chroma_client.get_or_create_collection(
    name="ncert_documents",
    metadata={"description": "NCERT PDFs and study notes"}
)

print("✅ ChromaDB initialized!")
print(f"   Current documents in DB: {collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Initializing ChromaDB...
✅ ChromaDB initialized!
   Current documents in DB: 0


## Step 5: PDF Text Extraction

In [5]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract text from a PDF file"""
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page_num, page in enumerate(reader.pages, 1):
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        print(f"❌ Error reading {pdf_path.name}: {e}")
        return ""

print("✅ PDF extraction function defined!")

✅ PDF extraction function defined!


## Step 6: Text Chunking Function

In [6]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Split text into overlapping chunks"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    
    return chunks

# Test the chunking function
test_text = "This is a test. " * 100
test_chunks = chunk_text(test_text, chunk_size=20, overlap=5)
print(f"✅ Chunking function defined!")
print(f"   Test: 100 words → {len(test_chunks)} chunks")

✅ Chunking function defined!
   Test: 100 words → 27 chunks


## Step 7: Check Available PDFs


In [7]:
pdf_files = list(DATA_FOLDER.glob("*.pdf"))

if pdf_files:
    print(f"📄 Found {len(pdf_files)} PDF file(s):")
    for pdf in pdf_files:
        size_mb = pdf.stat().st_size / (1024 * 1024)
        print(f"   • {pdf.name} ({size_mb:.2f} MB)")
else:
    print("⚠️  No PDF files found in data/ folder")
    print(f"   Please add PDF files to: {DATA_FOLDER.absolute()}")

📄 Found 1 PDF file(s):
   • Box-Cricket-Rules.pdf (0.48 MB)


## Step 8: Process PDFs and Upload to ChromaDB

This is where the magic happens! We'll:
1. Extract text from each PDF
2. Chunk the text
3. Generate embeddings
4. Store in ChromaDB

In [8]:
if not pdf_files:
    print("⚠️  Please add PDF files to the data/ folder first!")
else:
    print("🚀 Starting PDF processing...\n")
    
    all_chunks = []
    all_metadatas = []
    all_ids = []
    
    for pdf_file in pdf_files:
        print(f"📄 Processing: {pdf_file.name}")
        
        # Extract text
        text = extract_text_from_pdf(pdf_file)
        if not text.strip():
            print(f"   ⚠️  No text extracted")
            continue
        
        word_count = len(text.split())
        print(f"   ✓ Extracted {word_count:,} words")
        
        # Chunk text
        chunks = chunk_text(text)
        print(f"   ✓ Created {len(chunks)} chunks")
        
        # Prepare metadata
        for idx, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            all_metadatas.append({
                "source": pdf_file.name,
                "chunk_id": idx,
                "total_chunks": len(chunks)
            })
            all_ids.append(f"{pdf_file.stem}_{idx}")
    
    if all_chunks:
        print(f"\n🧠 Generating embeddings for {len(all_chunks)} chunks...")
        embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)
        
        print("💾 Uploading to ChromaDB...")
        collection.add(
            embeddings=embeddings.tolist(),
            documents=all_chunks,
            metadatas=all_metadatas,
            ids=all_ids
        )
        
        print(f"\n✅ Success! Uploaded {len(all_chunks)} chunks from {len(pdf_files)} PDF(s)")
        print(f"   Total documents in DB: {collection.count()}")
    else:
        print("\n⚠️  No content to upload")

🚀 Starting PDF processing...

📄 Processing: Box-Cricket-Rules.pdf
   ✓ Extracted 693 words
   ✓ Created 2 chunks

🧠 Generating embeddings for 2 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


💾 Uploading to ChromaDB...

✅ Success! Uploaded 2 chunks from 1 PDF(s)
   Total documents in DB: 2


## Step 9: Semantic Search Function

Retrieve relevant chunks based on query similarity.

In [9]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(
    path="chroma_db",
    settings=Settings(anonymized_telemetry=False)
)

collection = chroma_client.get_collection("ncert_documents")

data = collection.get()

print(f"Total chunks: {len(data['documents'])}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Total chunks: 2


In [10]:

# print all chunk
for i, doc in enumerate(data["documents"]):
    print("\n" + "=" * 80)
    print(f"Chunk {i+1}")
    print("=" * 80)
    print(doc)

    if data.get("metadatas"):
        print("\nMetadata:")
        print(data["metadatas"][i])


Chunk 1
Ahmedabad Branch of WICASA Box Cricket League Rules of the tournament 1. Each team shall include 7 players + 1 substitute in a team. 2. There will be 7 overs played per innings. The same may be reduced due to unforeseen circumstances like weather, dew, bad light etc. However minimum number of overs cannot be less than 1. 3. Every match shall be completed in 30 minutes (i.e 15 minutes per innings.) 4. Only CA students are eligible for participation in this tournament. If a team is found to include any player who is a non CA student then that team shall be disqualified and no refund will be availed to that team. 5. All matches will be day and night. Matches will be between 8AM to 11AM in morning and 7PM to 11PM in evening. The exact schedule will be circulated to all in a Whatsapp group of captains of each team on 17th September, 2020. The schedule will be prepared by picking of chits. 6. Scoring will be done on the application – Cric heroes. 7. Each team can have one substitute

In [ ]:
# print 5 chunks 
for i, doc in enumerate(data["documents"][:5]):
    print(f"\nChunk {i+1}")
    print(doc[:1000])  # first 1000 characters

In [ ]:
# inspect a chunk
chunk_no = 1

print(data["documents"][chunk_no])

print("\nMetadata:")
print(data["metadatas"][chunk_no])

In [ ]:
def retrieve_relevant_chunks(query: str, n_results: int = 5) -> Dict:
    """Retrieve relevant chunks for a query"""
    # Generate query embedding
    query_embedding = embedding_model.encode([query])[0]
    
    # Search in ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results
    )
    
    return results

print("✅ Retrieval function defined!")

## Step 10: Test Retrieval

Let's test if we can find relevant content!

In [ ]:
if collection.count() == 0:
    print("⚠️  No documents in database. Please run Step 8 first!")
else:
    # Test query
    test_query = "Founding of the Asiatic Society (Bengal) in which year ?"
    
    print(f"🔍 Test Query: {test_query}\n")
    results = retrieve_relevant_chunks(test_query, n_results=3)
    
    if results['documents'] and results['documents'][0]:
        print(f"✅ Found {len(results['documents'][0])} relevant chunks:\n")
        
        for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
            print(f"--- Chunk {i} (from {meta['source']}) ---")
            print(doc[:300] + "..." if len(doc) > 300 else doc)
            print()
    else:
        print("❌ No results found")

In [ ]:
data = collection.get()

chunk = data["documents"][30]

print(len(chunk))

## Step 11: RAG Question Answering Function

Combine retrieval with LLM generation!

In [ ]:
def ask_question(question: str, model: str = MODEL_NAME, n_results: int = 5):
    """Ask a question using RAG"""
    print(f"❓ Question: {question}\n")
    
    # Check if database has content
    if collection.count() == 0:
        print("❌ Database is empty. Please upload PDFs first!")
        return
    
    print("🔍 Retrieving relevant content...")
    results = retrieve_relevant_chunks(question, n_results=n_results)
    
    if not results['documents'] or not results['documents'][0]:
        print("❌ No relevant documents found.")
        return
    
    # Prepare context
    context = "\n\n".join(results['documents'][0])
    sources = set(meta['source'] for meta in results['metadatas'][0])
    print(f"✓ Found content from: {', '.join(sources)}\n")
    
    # Create prompt
    prompt = f"""Based on the following context from NCERT and study materials, answer the question accurately.

Context:
{context}

Question: {question}

Answer the question based only on the provided context. If the context doesn't contain enough information, say so."""
    
    # Query Ollama
    print(f"🤖 Generating answer using {model}...\n")
    print("💡 Answer:")
    print("-" * 60)
    
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            stream=True
        )
        
        full_answer = ""
        for chunk in response:
            content = chunk['message']['content']
            print(content, end='', flush=True)
            full_answer += content
        
        print("\n" + "-" * 60)
        print(f"\n📚 Sources: {', '.join(sources)}")
        
        return full_answer
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("   Make sure Ollama is running: 'ollama serve'")
        print(f"   And model is installed: 'ollama pull {model}'")

print("✅ Question answering function ready!")

## Step 12: Ask Your Questions! 🎯

Now you can ask questions about your PDFs!

In [ ]:
# Ask your first question!
ask_question("Tell me the time of match and also twhere we can see scores")

In [ ]:
# Try another question
ask_question("How Sarim become entrepenuer")

In [ ]:
# Ask your own question here!
my_question = ""  # Type your question here
if my_question:
    ask_question(my_question)

## Step 13: Database Statistics

In [ ]:
print("📊 Database Statistics:\n")
count = collection.count()
print(f"   Total chunks: {count}")

if count > 0:
    sample = collection.get(limit=count)
    sources = set(meta['source'] for meta in sample['metadatas'])
    print(f"   Unique documents: {len(sources)}")
    print(f"\n   Documents:")
    for source in sorted(sources):
        chunks_from_source = sum(1 for meta in sample['metadatas'] if meta['source'] == source)
        print(f"      • {source}: {chunks_from_source} chunks")

## Step 14: Clear Database (Optional)

Run this only if you want to start fresh!

In [ ]:
# Uncomment to clear the database
chroma_client.delete_collection("ncert_documents")
collection = chroma_client.get_or_create_collection(
    name="ncert_documents",
    metadata={"description": "NCERT PDFs and study notes"}
)
print("✅ Database cleared!")

#print("⚠️  Clear database code is commented out for safety")